# Final Project — Customer Review Sentiment and Key Concerns (Use Case #7)
**Goal:** Read one customer review. Find all aspects inside it. Tell the feeling for each aspect. Give one overall feeling: Positive, Negative, Neutral, or Mixed.

**Example:**
- Review: "The product is excellent but delivery was terrible."
- Output: product -> Positive, delivery -> Negative, Overall -> Mixed

**Rule:** Do not use a fixed list of aspects. The model must learn to find aspects by itself.

**Data:** Public human-labeled data from Hugging Face (SilvioLima/raw_data, DMASTE part, 7,524 reviews).


## Step 0 — Setup
Install once in Colab. This gets the tools we need.

In [ ]:
# Step 0: Install tools (run once)
!pip install -q datasets transformers scikit-learn torch pandas


### Imports
We bring all tools to one place. Each line has a simple comment.

In [ ]:
# Imports — what each tool does
import ast                  # Turn string "[('a','b')]" into a real list
import re                   # Find words in text
import pandas as pd         # Work with tables
from datasets import load_dataset  # Load public data
from sklearn.model_selection import train_test_split  # Split data safely
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report  # Scores
print("Imports ready")


---
## Section 1 — Explore the Data (EDA)
*First, look at the data. Do not build a model before you understand the data.*


In [ ]:
# 1.1 Load the public data
def load_raw_data():
    # Load from Hugging Face
    dataset = load_dataset("SilvioLima/raw_data")
    df = dataset["train"].to_pandas()
    return dataset, df

dataset, df = load_raw_data()
print(dataset)
print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()


In [ ]:
# 1.2 Check where the data comes from
def show_source_counts(df):
    # Count how many rows belong to each source
    counts = df["source"].value_counts()
    print(counts)
    return counts

show_source_counts(df)


In [ ]:
# 1.3 Look at one real example
def show_one_example(df, idx=0):
    # Show the sentence and its labels
    print("Sentence:")
    print(df.iloc[idx]["sentence"])
    print()
    print("Triples (aspect, opinion, feeling):")
    print(df.iloc[idx]["triples"])

show_one_example(df, 0)


In [ ]:
# 1.4 Look at five examples to see the pattern
def show_five_examples(df):
    for i in range(5):
        print(f"--- Review {i+1} ---")
        print(df.iloc[i]["sentence"][:160])
        print("Triples:", df.iloc[i]["triples"])
        print()

show_five_examples(df)


### Keep only DMASTE — this is the data for our task

In [ ]:
# 1.5 Keep DMASTE only (7,524 reviews)
def filter_dmast(df):
    dmast = df[df["source"] == "DMASTE"].copy()
    dmast = dmast[["sentence", "triples"]].copy()
    return dmast

dmast_df = filter_dmast(df)
print("DMASTE reviews:", len(dmast_df))
dmast_df.head()


In [ ]:
# 1.6 Change triples from string to list
def convert_triples(dmast_df):
    # Before: string, After: list
    print("Before type:", type(dmast_df.iloc[0]["triples"]))
    dmast_df["triples"] = dmast_df["triples"].apply(ast.literal_eval)
    print("After type:", type(dmast_df.iloc[0]["triples"]))
    return dmast_df

dmast_df = convert_triples(dmast_df)
print(dmast_df.iloc[0]["triples"][:2])


In [ ]:
# 1.7 How many aspects per review?
def count_triplets(dmast_df):
    counts = dmast_df["triples"].apply(len)
    print("Average per review:", round(counts.mean(), 2))
    print("Max in one review:", counts.max())
    print(counts.describe())
    return counts

count_triplets(dmast_df)


### Make one row per aspect — easy for the model

In [ ]:
# 1.8 Flatten the data
def flatten_data(dmast_df):
    rows = []
    for _, row in dmast_df.iterrows():
        sentence = row["sentence"]
        for aspect, opinion, sentiment in row["triples"]:
            rows.append({
                "text": sentence,
                "aspect": aspect,
                "opinion": opinion,
                "sentiment": sentiment
            })
    return pd.DataFrame(rows)

training_df = flatten_data(dmast_df)
print("Total rows (with hidden aspects):", len(training_df))
print(training_df.head())
print(training_df["sentiment"].value_counts())
print(training_df["sentiment"].value_counts(normalize=True).mul(100).round(2))


In [ ]:
# 1.9 Check text length
def check_text_length(df):
    df["text_len"] = df["text"].apply(len)
    print(df["text_len"].describe())
    # Most reviews are 160 to 595 characters. BERT will cut at 128 words.

check_text_length(training_df)


In [ ]:
# 1.10 Hidden vs visible aspects
def check_hidden_aspects(df):
    # aspect == -1 means hidden (not written clearly)
    visible = df[df["aspect"] != -1]
    hidden = df[df["aspect"] == -1]
    print("Visible (word is in text):", len(visible))
    print("Hidden (word is not in text):", len(hidden))
    print(hidden["sentiment"].value_counts())
    return visible, hidden

visible_df, hidden_df = check_hidden_aspects(training_df)


---
## Section 2 — Cleaning
*Remove parts the model cannot learn from.*


In [ ]:
# 2.1 Remove hidden aspects
def remove_hidden_aspects(df):
    # Hidden has no word to point to. Token model cannot learn it.
    print("Before:", len(df))
    clean = df[df["aspect"] != -1].copy()
    clean["text"] = clean["text"].astype(str).str.strip()
    clean["aspect"] = clean["aspect"].astype(str).str.strip()
    clean["opinion"] = clean["opinion"].astype(str).str.strip()
    clean = clean[clean["text"] != ""].reset_index(drop=True)
    print("After:", len(clean))
    return clean

training_df = remove_hidden_aspects(training_df)
print(training_df["sentiment"].value_counts())


In [ ]:
# 2.2 Summary of cleaning
def cleaning_summary(df):
    print("Cleaning done!")
    print("Hidden removed: 11,945")
    print("Final rows:", len(df))
    for lab in ["POS", "NEG", "NEU"]:
        print(lab, ":", len(df[df["sentiment"] == lab]))

cleaning_summary(training_df)
print("POS is 79%. So accuracy alone can fool us. We must watch F1.")


---
## Section 3 — Split into Train, Validation, Test
*Split by review, not by row. This stops the same review from being in two groups.*


In [ ]:
# 3.1 Split by unique review
def split_by_review(df):
    unique = df["text"].unique()
    print("Unique reviews:", len(unique))
    train_texts, test_texts = train_test_split(unique, test_size=0.2, random_state=42)
    train_texts, val_texts = train_test_split(train_texts, test_size=0.2, random_state=42)
    print("Train reviews:", len(train_texts))
    print("Val reviews:", len(val_texts))
    print("Test reviews:", len(test_texts))
    return train_texts, val_texts, test_texts

train_texts, val_texts, test_texts = split_by_review(training_df)


In [ ]:
# 3.2 Make the three tables
def make_splits(df, train_texts, val_texts, test_texts):
    train = df[df["text"].isin(train_texts)].reset_index(drop=True)
    val = df[df["text"].isin(val_texts)].reset_index(drop=True)
    test = df[df["text"].isin(test_texts)].reset_index(drop=True)
    return train, val, test

train_df, val_df, test_df = make_splits(training_df, train_texts, val_texts, test_texts)
print("Train rows:", len(train_df))
print("Val rows:", len(val_df))
print("Test rows:", len(test_df))


In [ ]:
# 3.3 Check for leakage (should be 0)
def check_leakage(train_texts, val_texts, test_texts):
    print("Train and Val overlap:", len(set(train_texts) & set(val_texts)))
    print("Train and Test overlap:", len(set(train_texts) & set(test_texts)))
    print("Val and Test overlap:", len(set(val_texts) & set(test_texts)))
    print("All 0 means no leakage. Good!")

check_leakage(train_texts, val_texts, test_texts)


---
## Section 4 — Turn Words into Tokens and Labels
*BERT reads tokens, not words. We give each token a label.*


In [ ]:
# 4.1 Load the tokenizer
from transformers import AutoTokenizer

def load_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    print("Tokenizer: bert-base-uncased")
    return tokenizer

tokenizer = load_tokenizer()

# What is offset? It maps token back to letters in the text.
def show_offsets(tokenizer, text):
    enc = tokenizer(text, return_offsets_mapping=True)
    toks = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    for t, (s, e) in zip(toks[:10], enc["offset_mapping"][:10]):
        print(f"  {t:12} -> '{text[s:e]}'  offset=({s},{e})")

show_offsets(tokenizer, train_df.iloc[0]["text"])


In [ ]:
# 4.2 Define the 5 labels
def define_labels():
    labels = ["O", "ASPECT", "OPINION_POS", "OPINION_NEG", "OPINION_NEU"]
    # O = nothing, ASPECT = product/battery
    # OPINION_* = excellent/terrible plus feeling
    label2id = {lab: i for i, lab in enumerate(labels)}
    id2label = {i: lab for lab, i in label2id.items()}
    print(label2id)
    return labels, label2id, id2label

LABELS, label2id, id2label = define_labels()


In [ ]:
# 4.3 Helper: Find where a word is in the text
def find_span(text, phrase):
    # Return start and end of phrase (lowercase search)
    phrase = str(phrase)
    start = text.lower().find(phrase.lower())
    if start == -1:
        return -1, -1
    end = start + len(phrase)
    return start, end

# Test
print(find_span("Battery is great", "Battery"))
print(find_span("Battery is great", "great"))


In [ ]:
# 4.4 Helper: Give label to one token
def label_one_token(start, end, asp_start, asp_end, opi_start, opi_end, sentiment, label2id):
    # Priority: opinion first, then aspect
    if start == 0 and end == 0:
        return label2id["O"]  # Special token [CLS]/[SEP]
    if opi_start != -1 and start >= opi_start and end <= opi_end:
        return label2id["OPINION_" + sentiment]
    if asp_start != -1 and start >= asp_start and end <= asp_end:
        return label2id["ASPECT"]
    return label2id["O"]

print("Helper ready")


In [ ]:
# 4.5 Main: Make labels for all tokens in one review
def make_token_labels(text, aspect, opinion, sentiment, tokenizer, label2id):
    # Step A: Get tokens and their positions
    enc = tokenizer(text, return_offsets_mapping=True, truncation=True, max_length=128)
    offsets = enc["offset_mapping"]
    # Step B: Find aspect and opinion spans
    asp_start, asp_end = find_span(text, aspect)
    opi_start, opi_end = find_span(text, opinion)
    # Step C: Label each token
    labels = []
    for s, e in offsets:
        lab = label_one_token(s, e, asp_start, asp_end, opi_start, opi_end, sentiment, label2id)
        labels.append(lab)
    return enc["input_ids"], enc["attention_mask"], labels

print("Main label function ready")


In [ ]:
# 4.6 Demo: See labels on one row
def demo_labels(df, idx, tokenizer, label2id, id2label):
    row = df.iloc[idx]
    ids, mask, labs = make_token_labels(row["text"], row["aspect"], row["opinion"], row["sentiment"], tokenizer, label2id)
    toks = tokenizer.convert_ids_to_tokens(ids)
    print("Text:", row["text"][:120])
    print("Aspect:", row["aspect"], "| Opinion:", row["opinion"], "| Feeling:", row["sentiment"])
    print("\nTokens with labels (only non-O):")
    for t, l in zip(toks, labs):
        if l != label2id["O"]:
            print(f"  {t:15} -> {id2label[l]}")

demo_labels(train_df, 2, tokenizer, label2id, id2label)


---
## Section 5 — Build the Dataset
*Make PyTorch datasets for training.*


In [ ]:
# 5.1 Convert one row to model input
import torch
from torch.utils.data import Dataset

def convert_row(row, tokenizer, label2id):
    # Use the label function
    text = row["text"]
    ids, mask, labs = make_token_labels(text, row["aspect"], row["opinion"], row["sentiment"], tokenizer, label2id)
    # Pad to 128 (all same length)
    enc = tokenizer(text, truncation=True, padding="max_length", max_length=128)
    labs_padded = labs + [label2id["O"]] * (128 - len(labs))
    labs_padded = labs_padded[:128]
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labs_padded}

# Test
print(convert_row(train_df.iloc[0], tokenizer, label2id)["input_ids"][:8])
print("Row conversion works")


In [ ]:
# 5.2 Convert all rows
def convert_all(df, tokenizer, label2id):
    all_tokens = []
    for _, row in df.iterrows():
        all_tokens.append(convert_row(row, tokenizer, label2id))
    return all_tokens

train_tokens = convert_all(train_df, tokenizer, label2id)
val_tokens = convert_all(val_df, tokenizer, label2id)
test_tokens = convert_all(test_df, tokenizer, label2id)
print("Train:", len(train_tokens), "Val:", len(val_tokens), "Test:", len(test_tokens))


In [ ]:
# 5.3 PyTorch Dataset class
class ReviewDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"]),
            "attention_mask": torch.tensor(item["attention_mask"]),
            "labels": torch.tensor(item["labels"])
        }

def build_datasets(train_tokens, val_tokens, test_tokens):
    return ReviewDataset(train_tokens), ReviewDataset(val_tokens), ReviewDataset(test_tokens)

train_dataset, val_dataset, test_dataset = build_datasets(train_tokens, val_tokens, test_tokens)
print("Datasets ready:", len(train_dataset), len(val_dataset), len(test_dataset))


---
## Section 6 — Model
*BERT that labels each token.*


In [ ]:
# 6.1 Check device (GPU or CPU)
import torch
from transformers import AutoModelForTokenClassification

def get_device():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)
    print("PyTorch:", torch.__version__)
    print("CUDA:", torch.cuda.is_available())
    return device

device = get_device()


In [ ]:
# 6.2 Load model
def load_model(num_labels, device):
    model = AutoModelForTokenClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
    model.to(device)
    print("Model on", device)
    return model

model = load_model(len(LABELS), device)


---
## Section 7 — Training
*Teach the model. 2 epochs for demo, 3-4 for full run.*


In [ ]:
# 7.1 Training settings
from transformers import TrainingArguments, Trainer

def get_training_args():
    args = TrainingArguments(
        output_dir="./bert_aste",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=2,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=50,
        learning_rate=2e-5,
        load_best_model_at_end=True
    )
    return args

args = get_training_args()
print(args)


In [ ]:
# 7.2 How to measure? Accuracy alone can trick us.
def compute_metrics(pred):
    preds = pred.predictions.argmax(-1).flatten()
    labels = pred.label_ids.flatten()
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

print("We will watch F1 (weighted). POS is 79%, so F1 is more honest than accuracy.")


In [ ]:
# 7.3 Build trainer
def build_trainer(model, args, train_dataset, val_dataset):
    trainer = Trainer(model=model, args=args, train_dataset=train_dataset, eval_dataset=val_dataset, compute_metrics=compute_metrics)
    print("Trainer ready. Run trainer.train() on T4 GPU (15-20 min)")
    return trainer

trainer = build_trainer(model, args, train_dataset, val_dataset)
# trainer.train()  # <-- Remove # to start training


In [ ]:
# 7.4 Save after training
def save_model(trainer, tokenizer):
    trainer.save_model("./bert_aste_final")
    tokenizer.save_pretrained("./bert_aste_final")
    print("Saved to ./bert_aste_final")

# save_model(trainer, tokenizer)  # Run after training
print("Save function ready")


---
## Section 8 — Scores and Overfit Check
*Is the model good? Is it memorizing (overfit) or not learning (underfit)?*


In [ ]:
# 8.1 Check validation scores
def show_val_scores(trainer):
    result = trainer.evaluate()
    print(result)
    # Example: {'eval_accuracy': 0.89, 'eval_precision': 0.87, 'eval_recall': 0.88, 'eval_f1': 0.875}
    return result

# show_val_scores(trainer)  # Run after training
print("Example: {'eval_accuracy': 0.89, 'eval_f1': 0.875}")


In [ ]:
# 8.2 Detailed report per label
def show_detailed_report(trainer, test_dataset):
    preds = trainer.predict(test_dataset)
    y_pred = preds.predictions.argmax(-1).flatten()
    y_true = preds.label_ids.flatten()
    print(classification_report(y_true, y_pred, target_names=LABELS, zero_division=0))

# show_detailed_report(trainer, test_dataset)  # Run after training
print("Example:")
print("  O            0.96  0.98  0.97")
print("  ASPECT       0.78  0.75  0.76  <- key for your aim")
print("  OPINION_POS  0.82  0.80  0.81")
print("  OPINION_NEG  0.75  0.70  0.72")
print("  OPINION_NEU  0.60  0.55  0.57")


### Which score to watch?
- **Do not trust accuracy alone.** If we guess POS always, we get 79% accuracy but learn nothing.
- **Watch F1 (weighted).** It balances precision and recall.
- **Watch ASPECT F1.** Finding the aspect is your main goal.
- **Watch OPINION_NEG recall.** If we miss negatives, we miss key problems.


In [ ]:
# 8.3 Overfit / Underfit check
def check_overfit(trainer):
    # Get train and val F1
    train_res = trainer.evaluate(eval_dataset=train_dataset)
    val_res = trainer.evaluate(eval_dataset=val_dataset)
    train_f1 = train_res.get("eval_f1", 0)
    val_f1 = val_res.get("eval_f1", 0)
    print(f"Train F1: {train_f1:.3f}")
    print(f"Val F1:   {val_f1:.3f}")
    gap = train_f1 - val_f1
    print(f"Gap: {gap:.3f}")
    if gap > 0.10:
        print("Warning: Overfit. Train much higher than Val. The model memorizes. Fix: add dropout, early stop, or more data.")
    elif val_f1 < 0.70 and train_f1 < 0.70:
        print("Warning: Underfit. Both low. The model is too simple or not trained long enough. Fix: train longer or lower learning rate.")
    else:
        print("Good balance. No strong overfit or underfit.")
    return train_f1, val_f1

# check_overfit(trainer)  # Run after training
print("Overfit check ready. Gap >0.10 means overfit.")


In [ ]:
# 8.4 If accuracy is too high (like 0.99), check this
def sanity_check_high_accuracy(trainer, test_dataset):
    # Very high can mean leakage (same review in train and test)
    # We already fixed leakage to 0, but double check
    print("If you see 0.99 accuracy:")
    print("1. Did you split by review? (We did, leakage 0)")
    print("2. Is the test set too easy? Check per-label F1, not just accuracy")
    print("3. Run check_overfit() above. Big gap means memorizing.")

print("Sanity check ready")


---
## Section 9 — Use the Model (No Fixed List)
*Give any review. The model finds aspects by itself.*


In [ ]:
# 9.1 Small helper: Clean token text
def clean_token(tok):
    return tok.replace("##", "")



In [ ]:
# 9.2 Small helper: Turn token predictions into lists
def decode_predictions(toks, preds, id2label):
    aspects = []
    opinions = []
    cur_text = ""
    cur_label = None
    for tok, lab in zip(toks, preds):
        if tok in ["[CLS]", "[SEP]", "[PAD]"]:
            continue
        lab_str = id2label[lab]
        is_sub = tok.startswith("##")
        tok_c = clean_token(tok)
        if lab_str == "ASPECT":
            if cur_label != "ASPECT":
                if cur_text:
                    if cur_label == "ASPECT": aspects.append(cur_text)
                    elif cur_label and cur_label.startswith("OPINION"): opinions.append((cur_text, cur_label))
                cur_text = tok_c
            else:
                cur_text += tok_c if is_sub else " " + tok_c
            cur_label = "ASPECT"
        elif lab_str.startswith("OPINION"):
            if cur_label != lab_str:
                if cur_text:
                    if cur_label == "ASPECT": aspects.append(cur_text)
                    elif cur_label and cur_label.startswith("OPINION"): opinions.append((cur_text, cur_label))
                cur_text = tok_c
            else:
                cur_text += tok_c if is_sub else " " + tok_c
            cur_label = lab_str
        else:
            if cur_text:
                if cur_label == "ASPECT": aspects.append(cur_text)
                elif cur_label and cur_label.startswith("OPINION"): opinions.append((cur_text, cur_label))
                cur_text = ""; cur_label = None
    if cur_text:
        if cur_label == "ASPECT": aspects.append(cur_text)
        elif cur_label and cur_label.startswith("OPINION"): opinions.append((cur_text, cur_label))
    return aspects, opinions

print("Decode helper ready")


In [ ]:
# 9.3 Small helper: Build triplets and overall feeling
def build_triplets(aspects, opinions):
    triplets = []
    for asp in aspects:
        if opinions:
            _, lab = opinions[0]
            sent = lab.split("_")[1]  # POS/NEG/NEU
        else:
            sent = "NEU"
        m = {"POS": "Positive", "NEG": "Negative", "NEU": "Neutral"}
        triplets.append({"aspect": asp, "sentiment": m[sent]})
    return triplets

def get_overall(triplets):
    pos = sum(1 for t in triplets if t["sentiment"] == "Positive")
    neg = sum(1 for t in triplets if t["sentiment"] == "Negative")
    if pos > 0 and neg > 0: return "Mixed"
    if pos > 0: return "Positive"
    if neg > 0: return "Negative"
    return "Neutral"

print("Triplet helpers ready")


In [ ]:
# 9.4 Main predict function (small, calls helpers above)
def predict_review(text):
    model.eval()
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc)
        preds = out.logits.argmax(-1)[0].cpu().tolist()
        ids = enc["input_ids"][0].cpu().tolist()
    toks = tokenizer.convert_ids_to_tokens(ids)
    aspects, opinions = decode_predictions(toks, preds, id2label)
    triplets = build_triplets(aspects, opinions)
    overall = get_overall(triplets)
    return {"review": text, "overall": overall, "aspects": triplets, "raw_aspects": aspects, "raw_opinions": opinions}

print("predict_review() ready")


In [ ]:
# 9.5 Try it (remove # after training)
# print(predict_review("The product is excellent but delivery was terrible."))
# print(predict_review("Battery drains fast but camera is amazing."))
# print(predict_review("The chair armrest is wobbly but fabric is comfortable."))
print("Demo ready. Training first, then uncomment above.")


---
## Section 10 — Any CSV (Chair, Phone, Any Product)
*The column name can be anything. We find it by itself.*


In [ ]:
# 10.1 Find the text column
import csv, io
from collections import Counter

def find_text_column(fieldnames):
    candidates = ["review_text", "review text", "review", "comment", "feedback", "text", "sentence"]
    norm = {h.lower().strip(): h for h in fieldnames}
    for c in candidates:
        if c in norm: return norm[c]
    for h in fieldnames:
        for c in candidates:
            if c in h.lower(): return h
    return fieldnames[0]

print("Column finder ready")


In [ ]:
# 10.2 Read any CSV and analyze
def analyze_csv_bytes(content: bytes):
    text = content.decode("utf-8")
    reader = csv.DictReader(io.StringIO(text))
    col = find_text_column(reader.fieldnames)
    print("Found column:", col)
    results = []
    for row in reader:
        txt = (row.get(col) or "").strip()
        if not txt: continue
        results.append(predict_review(txt))
    # For dashboard
    overall = Counter(r["overall"] for r in results)
    print("Overall:", dict(overall))
    all_a = [a["aspect"] for r in results for a in r["aspects"]]
    print("Top concerns:", Counter(all_a).most_common(5))
    return results

print("CSV analyzer ready")
print("Use: analyze_csv_bytes(open('chair.csv','rb').read())")


---
## Summary
- **EDA:** Saw data, removed hidden aspects (11,945), kept 16,288
- **Split:** By review, no leakage (0)
- **Labels:** 5 labels with clean helpers
- **Model:** BERT, safe for CPU/GPU
- **Training:** 2 epochs demo, watch F1 not just accuracy
- **Check:** Overfit if train F1 >> val F1 (gap >0.10)
- **Use:** Any review or any CSV, no fixed list

**Next:** Colab -> T4 GPU -> Run all -> trainer.train() -> check scores -> try Amazon reviews.
